# 3.1. Linear Regression
D2L의 Linear Regression 장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

## 1. 회귀 문제란?

회귀는 숫자를 예측하는 문제임.

예를 들어서
> 집값 예측 => 숫자로 함  
> 주식 가격 예측 => 숫자  
> 병원 입원 기간 예측 => 숫자  
> 판매량 예측 => 숫자  

반대로 분류는 고양이/강아지나 합/불, 스팸문자/스팸문자아님처럼 카테고리를 예측하는 문제이다.
D2L도 여기에서 회귀와 분류를 구분한다.


## 2. 선형 회귀의 가장 단순한 형태

예를 들어 
> 집값 = 면적 영향 + 나이 영향 + 기본값  
> price = w_area + w_age * age + b

w_area, w_age는 weight이다. 각 feature가 예측값에 얼마나 영향을 주는지 나타낸다. b는 bias 입력 feature가 모두 0일 때의 기본값 또는 절편 역할이다. D2L은 bias가 있어야 원점을 반드시 지나는 직선으로 제한되지 않고 더 일반적인 선형 함수를 표현할 수 있다고 설명한다.

> x = 입력  
> m = weight  
> b = bias  
> y = 예측값  

D2L 집값 예시는 feature가 2개라서 다음처럼 된다

> price = w_area * area + w_age * age + b

feature가 많아지면 이렇게 쓴다.

> ŷ = w1*x1 + w2*x2 + ... + wd*xd + b

ŷ는 y-hat이라고 읽고, 모델이 예측한 값이라는 뜻이다. 진짜 정답은 y 이고, 모델 예측값은 ŷ이다.
D2L도 hat 기호를 estimate (추정값)이라고 설명한다.

## 3. 벡터 표기

feature가 많아지면 수식을 하나하나 쓰기 어려워진다.

그래서 벡터로 묶는다

> x = [x1, x2, ..., xd]  
> w = [w1, w2, ..., wd]

그러면 이런식으로 줄어든다.

> ŷ = wᵀx + b

의미는

> wᵀx = w1*x1 + w2*x2 + ... + wd*xd

torch.dot(w, x) 와 같은 내적이다.
pyTorch로는 이렇다

```python
import torch

x = torch.tensor([100.0, 10.0])  # area, age
w = torch.tensor([3.0, -2.0])    # area weight, age weight
b = torch.tensor(50.0)

y_hat = torch.dot(w, x) + b
print(y_hat)
```

여기에서 w[0] = 3.0이라는 것은 면적이 커질수록 집값 예측이 증가한다는 뜻이다. w[1] = -2.0이라는 것은 나이가 많아질수록 집값 예측이 감소한다는 것이다. 이건 예시라 실제에서는 weight는 데이터로부터 학습된다.

## 4. 데이터 전체를 행렬로 보면

샘플 하나의 feature는 벡터 x다.
그런데 학습 데이터에는 샘플이 여러개 있다.

예를 들어서 집 3개의 데이터가 있다고 해보자.

> 집1: [면적, 나이]  
> 집2: [면적, 나이]  
> 집3: [면적, 나이]  

이걸 행렬로 묶으면 

> x = [  
    [x11, x12],  
    [x21, x22],  
    [x31, x32]  
]

여기에서 x shape = (샘플 수(n), feature 수(d))이다.

D2L에서는 이걸 design matrix라고 부른다. 각 행은 하나의 예제, 각 열은 하나의 feature다. 전체 데이터에 대한 예측은 ŷ = Xw + b로 표현된다. 여기서 b는 broadcasting으로 각 샘플 예측값에 더해진다.

In [ ]:
X = torch.tensor([
    [100.0, 10.0],
    [80.0, 5.0],
    [120.0, 20.0]
])  # shape: (3, 2)

w = torch.tensor([3.0, -2.0])  # shape: (2,)
b = torch.tensor(50.0)

y_hat = X @ w + b
print(y_hat)

## 5. Loss Function 예측이 얼마나 틀렸는가?

모델이 예측한 값 ŷ와 실제 정답 y가 같으면 좋은 것이다.

하지만 처음에는 거의 틀린다.

그래서 얼마나 틀렸는지를 숫자로 측정해야 한다. 이게 loss function이다.
D2L은 회귀 문제에서 가장 흔한 loss로 squared error, 제곱 오차를 사용한다.

> loss = 1/2 * (ŷ - y)^2

제곱을 하는 이유는 양수를 만들기 위해서이다.

예를 들어
예측값이 10, 정답이 7이면 오차 = 3
예측값이 4, 정답이 7이면 오차 = -3

둘 다 틀린 정도는 3이다. 제곱하면 둘 다 9가 된다.

1/2를 붙이는 이유는 loss = 1/2 * (ŷ - y)^2 식을 미분하면

d/dŷ [1/2 * (ŷ - y)^2] = ŷ - y 이런식으로 앞의 2가 1/2랑 사라진다. 계산을 깔끔하게 하기 위해 붙인 것이다.
D2L에서도 이 상수는 실제 의미를 바꾸지 않지만 미분할 때 편리하다고 설명하였다.

## 6. 전체 데이터의 loss

샘플 하나의 loss는 다음과 같다

> l = 1/2 * (ŷ - y)^2

데이터가 여러개 있으면 각 샘플의 loss를 평균낸다.

> L(w, b) = 전체 샘플 loss의 평균  
> L(w, b) = 1/n * Σ 1/2 * (wᵀxᶦ + b - yᶦ)^2

모든 데이터에 대해서 예측값 - 정답을 구하고, 제곱하고 1/2를 곱하고 평균을 낸다.

목표는 이 전체 loss를 가장 작게 만드는 w와 b를 찾는 것이다.
D2L에서도 학습 목표를 L(w, b)를 최소화하는 파라미터를 찾는 것으로 정의한다.

## 7. 학습이란 무엇인가?

학습 = w와 b를 조금씩 바꿔서 loss를 줄이는 과정

처음에는 w와 b를 랜덤으로 잡는다.

> w = torch.randn(2, requires_grad=True)  
> b = torch.zeros(1, requires_grad=True)

처음 예측은 좋지 않을 것이다.



In [ ]:
# X: 입력 feature
# 각 열은 [면적, 나이]
X = torch.tensor([
    [100.0, 10.0],
    [80.0, 5.0],
    [120.0, 20.0],
    [90.0, 8.0]
])

# y: 실제 정답 label
# 각 집의 실제 가격이라고 생각하면 됨
y = torch.tensor([330.0, 280.0, 370.0, 300.0])

# w: feature 2개에 대한 weight
w = torch.randn(2, requires_grad=True)

# b: bias
b = torch.zeros(1, requires_grad=True)

# 예측값
y_hat = X @ w + b

# loss 계산
loss = ((y_hat - y) ** 2).mean() / 2

print("w:", w)
print("b:", b)
print("y_hat:", y_hat)
print("y:", y)
print("loss:", loss)

In [ ]:
# loss.backward() # 미분한다

print("w.grad:", w.grad) # loss를 줄이기 위해 w를 어느 방향으로 바꿔야 하는지
print("b.grad:", b.grad) # loss를 줄이기 위해 b를 어느 방향으로 바꿔야 하는지

In [ ]:
lr = 0.0001

with torch.no_grad():
    w -= lr * w.grad # 업데이트 해주는 부분
    b -= lr * b.grad # 업데이트 해주는 부분

    w.grad.zero_() # grad를 0으로 초기화 해주는 부분 (누적되기 때문에 초기화 해야함)
    b.grad.zero_() # grad를 0으로 초기화 해주는 부분

print("updated w:", w)
print("updated b:", b)

In [ ]:
import torch

X = torch.tensor([
    [100.0, 10.0],
    [80.0, 5.0],
    [120.0, 20.0],
    [90.0, 8.0]
])

y = torch.tensor([330.0, 280.0, 370.0, 300.0])

w = torch.randn(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.0001
epochs = 100

for epoch in range(epochs):
    y_hat = X @ w + b
    loss = ((y_hat - y) ** 2).mean() / 2

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

        w.grad.zero_()
        b.grad.zero_()

    if epoch % 10 == 0:
        print(f"epoch {epoch}, loss: {loss.item():.4f}")

print("learned w:", w)
print("learned b:", b)

전체 학습 루프는 이렇게 된다.

D2L에서 말하는 gradient descent도 이 흐름이라고 한다. loss를 줄이는 방향으로 파라미터를 반복적으로 업데이트 한다.

## 8. 왜 gradient의 반대 방향으로 가는가?

> gradient = loss가 가장 빠르게 증가하는 방향

그런데 우리는 loss값을 줄이고 싶다. 그래서 gradient 반대 방향으로 가는것이다

> w ← w - learning_rate * gradient

현재 w에서 loss가 증가하는 방향을 빼면 loss가 감소하는 방향으로 이동한다.
D2L의 minibatch SGD 업데이트 식도 이 구조이다. minibatch에서 계산한 gradient에 `learning rate η`를 곱하고 현재 파라미터에서 뺀다.

## 9. Minibatch SGD

전체 데이터를 한 번에 다 써서 gradient를 계산할 수도 있다.

> Full batch gradient descent = 전체 데이터를 보고 한 번 업데이트

하지만 이건 데이터가 크면 느리다. 반대로 하나만 보고 업데이트 할 수도 있다.

> SGD = 샘플 하나를 보고 한 번 업데이트

하지만 이건 불안정하고 비효율적일 수도 있다. 그래서 중간으로 한다.

> Minibatch SGD = 데이터 일부 묶음만 보고 업데이트

예를 들어서 batch size가 32라면  
데이터 32개를 뽑고  
32개 평균 loss를 계산하고  
gradient 계산  
w, b 업데이트  
또 다음 32개 뽑고  
반복한다.  

D2L은 minibatch 크기가 메모리, 가속기 수, layer 종류, 전체 데이터 크기 등에 따라 달라지지만 32~256정도, 가능하면 2의 큰 거듭제곱 배수를 시작점으로 잡을 수 있다고 한다.

## 10. Hyperparameter란?

D2L에서 나오는 용어이다. 이건 학습으로 자동으로 업데이트되는 값이 아니다.

예를 들어서 learning rate, batch size, epoch 수 이다.

반대로 w, b는 학습되는 값이다.

#### parameter:
- w
- b

#### hyperparameter:
- learning rate
- batch size
- epoch

모델이 학습한다라고 말할 때 실제로 학습되는 건 w, b같은 parameter이고 learning rate같은 건 우리가 정하는 설정값이다.
D2L에서도 minibatch size와 learning rate처럼 학습 루프에서 직접 업데이트되지 않는 조정 가능한 값을 hyperparameter라고 설명한다.

## 11. Prediction

학습이 끝나면 최종적으로 학습된 w, 학습된 b를 얻게 될 것이다.

D2L은 이걸 ŵ, b̂ 이렇게 표기한다.

이제 새로운 집이 들어오면

> 새 집 feature x = [면적, 나이]
> 예측값 = ŵᵀx + b̂

이게 prediction이다.

딥러닝에서는 이 과정을 inference라고도 부른다고 한다. D2L에서는 용어 혼동을 줄이기 위해 가능하면 prediction으로 사용한다고 한다.

## 12. Vectorization: for문보다 행렬 연산을 써라

D2L은 속도 때문에 vectorization을 강조한다.

```py
for i in range(n): # 나쁜 방식
    c[i] = a[i] + b[i]
```

```py
c = a + b # 좋은 방식
```

```py
y_hat_list = [] # 나쁜 방식

for i in range(len(X)):
    y_hat_i = torch.dot(X[i], w) + b
    y_hat_list.append(y_hat_i)
```

```py
y_hat = X @ w + b # 좋은 방식
```

D2L은 전체 minibatch를 동시에 처리하려면 계산을 vectorize하고 Python for-loop대신에 빠른 선형대수 라이브러리를 활용해야한다고 설명했다.
실제 예시에서도 elementwise + 연산이 Python for-loop보다 훨씬 빠르다는 걸 보여준다.

## 13. 왜 squared loss를 쓰는가?

D2L은 squared loss를 확률 관점에서도 설명해준다.

가정은
> 실제 y는 완벽한 선형식이 아니라 noise가 섞인 값이다.

수식
> y = wᵀx + b + ε

여기에서 `ε`는 noise다
D2L은 이 noise가 평균 0, 분산 σ²인 정규분포를 따른다고 가정한다.

> ε ~ N(0, σ²)
이건 진짜 데이터는 대체로 선형 관계를 따르지만 측정 오차나 설명되지 않는 요인 때문에 약간씩 흔들린다. 그 흔들림이 정규분포 형태라고 가정한다.

이 가정을 하면 정규분포 noise를 가정하고 최대가능도추정(MLE)을 하면 결국 squared error를 최소화하는 것과 같아진다.

오차가 정규분포 noise라고 가정하면,
MSE를 최소화하는 것이 확률적으로 가장 그럴듯한 파라미터를 찾는 것과 연결된다.

D2L은 고정된 σ를 가정하면 negative log-likelihood의 핵심 항이 squared error와 같아지고, 평균제곱오차 최소화가 additive Gaussian noise 가정 아래 선형 모델의 maximum likelihood estimation과 동등하다고 설명한다.

## 14. 선형 회귀는 신경망인가?

D2L에선 선형 회귀를 신경망 관점으로도 본다.

입력 feature가 있다고 했을때 x1, x2, ..., xd

각 입력은 output neuron 하나에 직접 연결된다.

> o = w1*x1 + w2*x2 + ... + wd*xd + b

출력이 하나의 숫자이기 때문에 output neuron도 하나이다.

그래서 선형 회귀는 다음처럼 볼 수 있다. 

> 입력층 => 출력층

중간 hidden layer가 없다. 그래서 선형 회귀는 아주 단순한 `single-layer fully connected neural network`로 볼 수 있다. D2L도 선형 회귀를 입력이 출력에 직접 연결된 단일층 신경망으로 해석할 수 있다고 설명한다.

In [ ]:
import torch

# 1. 데이터 준비
# X: feature matrix, shape = (n, d)
# y: label vector, shape = (n,)
X = torch.tensor([
    [100.0, 10.0],
    [80.0, 5.0],
    [120.0, 20.0],
    [90.0, 8.0]
])

y = torch.tensor([330.0, 280.0, 370.0, 300.0])

# 2. 파라미터 초기화
w = torch.randn(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 3. 하이퍼파라미터
lr = 0.0001
epochs = 100

# 4. 학습
for epoch in range(epochs):
    # 예측
    y_hat = X @ w + b

    # loss 계산
    loss = ((y_hat - y) ** 2).mean() / 2

    # gradient 계산
    loss.backward()

    # 파라미터 업데이트
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

        # gradient 초기화
        w.grad.zero_()
        b.grad.zero_()

    if epoch % 10 == 0:
        print(epoch, loss.item())

print("learned w:", w)
print("learned b:", b)

이 코드는 D2L 3.1의 핵심 대부분이다. 위 예시는 feature scale이 크고 데이터가 적어 학습은 불안정하다.
다음장에서 synthetic data, scratch implementation, concise implementation으로 더 체계적으로 다룬다.

## 15. 오늘의 정리

- 회귀는 숫자를 예측하는 문제다.
- 선형 회귀는 feature들의 가중합으로 label을 예측한다.
- 예측식은 ŷ = wᵀx + b 이다.
- w는 각 feature의 영향력이다.
- b는 bias, 즉 절편이다.
- 데이터 전체에 대해서는 ŷ = Xw + b 로 쓴다.
- loss는 예측값과 정답의 차이를 숫자로 나타낸다.
- 회귀에서 가장 흔한 loss는 squared error다.
- squared error는 1/2 * (ŷ - y)^2 형태다.
- 학습은 loss를 줄이도록 w와 b를 업데이트하는 과정이다.
- gradient는 loss가 증가하는 방향이다.
- loss를 줄이려면 gradient의 반대 방향으로 이동한다.
- minibatch SGD는 데이터 일부를 뽑아서 gradient를 계산하고 업데이트하는 방식이다.
- learning rate와 batch size는 hyperparameter다.
- 학습된 w와 b를 사용해서 새로운 데이터의 값을 예측한다.
- Python for-loop보다 벡터화된 행렬 연산을 써야 빠르다.
- 정규분포 noise를 가정하면 squared loss는 최대가능도추정과 연결된다.
- 선형 회귀는 입력이 출력 하나에 직접 연결된 단일층 신경망으로 볼 수 있다.

정리하면

입력 x
→ 예측 ŷ = Xw + b
→ 정답 y와 비교
→ loss 계산
→ loss.backward()
→ w, b 업데이트
→ 반복

이 구조는 MLP, CNN, RNN, Transformer에서도 거의 그대로 유지된다. 바뀌는 것은 ŷ = Xw + b처럼 단순한 모델이 아니라, 훨씬 복잡한 neural network가 들어간다는 점이다. D2L도 이 장의 요약에서 선형 회귀가 parameterized form, differentiable objective, minibatch SGD, unseen data 평가 등 이후 모델에 공통으로 필요한 구성요소를 소개한다고 정리한다.

### Feature, Label, x, X, y 구분

feature는 모델이 입력으로 사용하는 정보다. 집값 예측에서는 면적과 나이가 feature다.

label은 모델이 맞춰야 하는 정답이다. 집값 예측에서는 실제 집값이 label이다.

샘플 하나의 feature 값들을 벡터로 모으면 x가 된다.

예를 들어 첫 번째 집의 면적이 100, 나이가 10이면:

x = [100, 10]

여기서 x 안의 원소 하나하나가 feature 값이다.

전체 샘플의 feature 벡터를 모두 모으면 X가 된다.

X = [
  [100, 10],
  [80, 5],
  [120, 20],
  [90, 8]
]

각 샘플의 label 값을 모으면 y가 된다.

y = [330, 280, 370, 300]

따라서 X는 입력 데이터 전체이고, y는 정답 데이터 전체다.